# Competitive strategy research — group MaRs-777 (Police)

**This notebook explains and displays. It computes nothing of its own.**

Every statistic shown here is produced by tested functions in `research/`,
and every number is read from result files that were committed when the
experiment ran. That split is deliberate: a notebook cell is not covered by
the test suite, so no business or statistical logic is allowed to live only
here. If a number in this notebook is wrong, a test in `tests/research/`
should already be failing.

It also **runs no games**. Opening it does not replay a benchmark, does not
touch the network, needs no credential, and cannot reach the sealed final
holdout — it reads the one recorded result of the single evaluation that was
permitted.


## 0. Reproducing this notebook

```bash
uv sync --group notebook          # resolves Jupyter locally
uv run jupyter nbconvert --execute --to notebook --inplace \
    notebooks/competitive_research.ipynb
```

**Jupyter is an optional dependency group, and it is locked.** `uv.lock`
resolves it, so `uv sync --group notebook` is reproducible - but it is not a
*default* group, so plain `uv sync --frozen`, which is what CI runs, never
installs a notebook stack. A tournament agent does not need one to play.

You do not need Jupyter to check any number here. The notebook renders as
text on GitHub, and everything it displays regenerates without it:

```bash
uv run python -m research.candidate_main figures  --out results
uv run python -m research.candidate_main evidence --out results
```


In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"


def load(*parts):
    """Read one committed result document. No computation, no games."""
    return json.loads((RESULTS.joinpath(*parts)).read_text(encoding="utf-8"))


screening = load("candidates", "screening.json")
development = load("candidates", "full_C4.json")
validation = load("candidates", "validation_C4.json")
stress = load("candidates", "stress_C4.json")
holdout = load("candidates", "final_holdout_result.json")
freeze = load("candidates", "freeze_C4.json")
print("loaded committed evidence only - no game was played")

## 1. The frozen baseline

The shipped policy before any candidate existed. The headline is weighted
over the varied configurations and **excludes** the fixed Appendix-F
reference geometry, which has only 9 scenarios and is reported separately.


In [ ]:
overall = json.loads((RESULTS / "tables" / "overall.json").read_text(encoding="utf-8"))
print("statistical unit :", overall["statistical_unit"])
print("headline scenarios:", overall["headline_scenarios"])
print("raw rows          :", overall["raw_rows"])

## 2. The methodology correction that came first

The first benchmark was **invalid and is kept in the record**. Two defects
were found and fixed before any candidate was judged:

1. **A holdout that had been read is not a holdout.** Stage 9B-0 executed a
   bank called `holdout` and then read its results while ranking candidate
   ideas. It was honestly reclassified as **validation**, and a genuinely
   sealed `final_holdout` was created, enumerated, hashed and committed
   *before any candidate existed*.
2. **A row is not an observation.** Duplicate deterministic scenarios were
   being bootstrapped as if independent. The unit became `scenario_id`, and
   the published headline moved from 0.0526 to **0.0638** as a result.

Both old numbers remain in `docs/research/COMPETITIVE_RESEARCH.md` beside the
corrected ones. Silently replacing a published number is the same error in a
different place.


## 3. Candidate hypotheses, and the two that failed

Four candidates, each one conceptual change, all frozen with a source hash
**before** being run. Screening membership was fixed by digest of the
scenario id alone, so no outcome could influence which scenarios judged it.


In [ ]:
for key in ("C1", "C2", "C3", "C4"):
    entry = screening[key]
    ci = entry["paired_ci"]
    print(
        f"{key:<3} {entry['summary'][:52]:<52} {ci['mean']:+.4f}"
        f"  [{ci['ci_low']:+.4f}, {ci['ci_high']:+.4f}]"
    )

**C1 refutes the pursuit hypothesis outright.** A belief-directed *mover*
alone did not merely fail to help — it lost **every one of the 23 games the
shipped policy had won**, and barrier use collapsed from 3.40 to 0.40.
Chasing the evidence walks the police onto the hottest cell, which then
suppresses the very barriers its wins were made of.

**C3 refutes "spend more of the quota".** Lowering the floor to 0.3 was
*worse* than 0.9 and its interval includes zero. Both C2 and C3 place
**fewer** barriers than the baseline, not more.

**C4 is the ablation that reversed the ranking.** Running the same barrier
rule behind the *shipped* mover beat running it behind the new one — so the
mover change was not merely unnecessary, it cost about two points. Without
this ablation the study would have advanced C2 and attributed the gain to the
wrong half of the change.


## 4. The same frozen candidate on four independent banks

One source hash throughout. Development selected it; validation and stress
were never tuned on; the final holdout was sealed before it existed.


In [ ]:
banks = (
    ("development", development),
    ("validation", validation),
    ("stress", stress),
    ("FINAL HOLDOUT", holdout),
)
for name, doc in banks:
    one = doc["overall"]
    delta = one.get("delta", one.get("win_delta"))
    low, high = one.get("ci_low"), one.get("ci_high")
    if low is None:
        low, high = doc["paired_ci"]["ci_low"], doc["paired_ci"]["ci_high"]
    print(f"{name:<14} N={one['n']:<5} {delta:+.4f}  [{low:+.4f}, {high:+.4f}]")

**Read the last line once.** The final holdout was evaluated exactly once,
and that number can never be improved by running it again — the tooling
refuses a second run for the same commitment and candidate.


## 5. Confidence intervals, and what they are not

Intervals are a **deterministic bootstrap over paired per-scenario
differences**, keyed by `scenario_id`; a comparison is refused unless both
sides played exactly the same scenario set. A cell below the minimum sample
gets **no interval at all** rather than a falsely narrow one.

They describe sampling variation **within this corpus**. They say nothing
about an unknown external opponent.


In [ ]:
print("holdout family results (all seven improved):")
for name, cell in sorted(holdout["family"].items(), key=lambda one: -one[1]["delta"]):
    print(
        f"  {name:<20} N={cell['n']:<4} {cell['delta']:+.4f}"
        f"  [{cell['ci_low']:+.4f}, {cell['ci_high']:+.4f}]"
    )
print()
print("sparse reference cell, reported separately and given no weight:")
cell = holdout["config"]["appendixF-example"]
print(f"  appendixF-example N={cell['n']}  ci={cell['ci_low']}")

## 6. The sealed holdout

A holdout only means something if nobody looked. The commitment is a plain
SHA-256 over the canonical scenario list, recorded **before** any candidate
existed — anyone can recompute it; nobody can quietly change which scenarios
the promotion was going to be judged on.


In [ ]:
seal = freeze["final_holdout"]
print("commitment      ", seal["commitment_sha256"])
print("scenario count  ", seal["count"])
print("candidate frozen", freeze["candidate_sha256"])
print("holdout matches ", holdout["commitment_sha256"] == seal["commitment_sha256"])
print()
print("the freeze record still says, correctly, that at the time it was written:")
print("  final_holdout_evaluated =", freeze["final_holdout_evaluated"])
print("  production_promotion    =", freeze["production_promotion"])

## 7. Why there is no learning curve

**Nothing in this project is trained.** There is no model, no parameter
update, no epoch, no gradient and no reward signal. Presenting a "training
loss" or a "neural learning curve" would be a fabricated result.

The truthful equivalent is the **strategy research progression**: one point
per candidate revision, in the order it was evaluated, each point a separate
replayed evaluation against a frozen baseline. The rejected candidates stay
on the curve — a progression showing only survivors would draw a rising line
out of a search that in fact went down twice.


In [ ]:
from IPython.display import Image, display

figures = RESULTS / "figures" / "candidates"
for name in ("strategy_research_progression.png", "c4_by_bank.png", "c4_final_holdout_family.png"):
    print(name)
    display(Image(filename=str(figures / name)))

## 8. The promotion decision

Every gate was frozen in `docs/research/COMPETITIVE_RESEARCH.md` §9 **before
any candidate existed**, including the numeric definition of a material
regression: a drop of more than five percentage points whose interval also
excludes zero.


In [ ]:
for bank, assessed in freeze["assessment"].items():
    print(
        bank,
        "passed =",
        assessed["passed"],
        "| material regressions:",
        assessed["material_regressions"] or "none",
    )
print()
one = holdout["overall"]
print(
    f"final holdout: {one['baseline_wins']} -> {one['candidate_wins']} wins, "
    f"gains {one['gains']}, losses {one['losses']}, delta {one['delta']:+.4f}"
)
print("legality failures:", holdout["legality_failures"])

**Promoted.** The exact frozen candidate became the production barrier rule,
proved behaviourally identical to the research candidate on an exhaustive
state matrix and on committed development, validation and stress games —
without replaying the holdout, which is consumed.

**What this claims:** a validated improvement across this project's own
frozen legal benchmark corpus, confirmed by a single pre-committed holdout.

**What it does not claim:** that any particular match will be won. The
external opponent is unknown, and the seven opponent families are models we
wrote, not the field.
